# 01. Khám phá dữ liệu thời tiết

Notebook này thực hiện:

- Đọc dữ liệu thời tiết gốc.
- Kiểm tra kích thước, kiểu dữ liệu và khoảng thời gian.
- Kiểm tra giá trị thiếu, dòng trùng và tần suất lấy mẫu.
- Khảo sát biến mục tiêu `rain`.
- Trực quan hóa chuỗi thời gian, phân phối và tương quan.

Notebook không thay đổi dữ liệu gốc.

## 1. Thư viện và cấu hình

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

DATA_PATH = Path("../data/raw/weather.csv")
TARGET_COL = "rain"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {DATA_PATH}. Hãy đặt dữ liệu gốc vào data/raw/."
    )

## 2. Đọc và kiểm tra dữ liệu tổng quan

In [ ]:
raw_df = pd.read_csv(DATA_PATH)
raw_df.columns = [str(col).strip() for col in raw_df.columns]

date_candidates = ["date", "datetime", "date time", "timestamp", "time"]
lower_map = {col.lower(): col for col in raw_df.columns}
date_col = next(
    (lower_map[name] for name in date_candidates if name in lower_map),
    raw_df.columns[0],
)

raw_df[date_col] = pd.to_datetime(raw_df[date_col], errors="coerce")
if raw_df[date_col].notna().mean() < 0.80:
    raw_df[date_col] = pd.to_datetime(
        raw_df[date_col], errors="coerce", dayfirst=True
    )

print("Kích thước dữ liệu:", raw_df.shape)
print("Cột thời gian:", date_col)
print("Khoảng thời gian:", raw_df[date_col].min(), "->", raw_df[date_col].max())
display(raw_df.head())

In [ ]:
summary_df = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing": raw_df.isna().sum(),
    "missing_percent": (raw_df.isna().mean() * 100).round(2),
    "unique": raw_df.nunique(dropna=True),
})
display(summary_df)

print("Số dòng thời gian không hợp lệ:", raw_df[date_col].isna().sum())
print(
    "Số mốc thời gian trùng:",
    raw_df.duplicated(subset=[date_col]).sum(),
)

## 3. Kiểm tra dữ liệu lấy mẫu đều

In [ ]:
valid_dates = (
    raw_df[date_col]
    .dropna()
    .drop_duplicates()
    .sort_values()
)
time_diffs = valid_dates.diff().dropna()

print("Các khoảng lấy mẫu phổ biến:")
display(time_diffs.value_counts().head(10).rename("count").to_frame())

expected_delta = pd.Timedelta("10min")
irregular_count = int((time_diffs != expected_delta).sum())
print("Số khoảng không bằng 10 phút:", irregular_count)

## 4. Thống kê mô tả và biến mục tiêu

In [ ]:
for col in raw_df.columns:
    if col != date_col:
        raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

display(raw_df.describe(include="all").T)

target_matches = [
    col for col in raw_df.columns
    if col.lower() == TARGET_COL.lower()
]
if not target_matches:
    raise KeyError(f"Không tìm thấy cột mục tiêu '{TARGET_COL}'.")
TARGET_COL = target_matches[0]

rain = raw_df[TARGET_COL].dropna()
print("Tỷ lệ không mưa:", (rain == 0).mean().round(4))
print("Tỷ lệ có mưa >= 0.1 mm:", (rain >= 0.1).mean().round(4))
print("Lượng mưa lớn nhất:", rain.max())

## 5. Trực quan hóa

In [ ]:
plot_df = (
    raw_df.dropna(subset=[date_col])
    .sort_values(date_col)
    .set_index(date_col)
)

fig, axes = plt.subplots(2, 1, figsize=(16, 9))

axes[0].plot(
    plot_df.index[:3000],
    plot_df[TARGET_COL].iloc[:3000],
    color="#28536B",
    linewidth=1,
)
axes[0].set_title("Lượng mưa theo thời gian - 3.000 quan sát đầu")
axes[0].set_xlabel("Thời gian")
axes[0].set_ylabel("Lượng mưa (mm)")

axes[1].hist(
    rain.clip(upper=rain.quantile(0.99)),
    bins=50,
    color="#D1495B",
    alpha=0.85,
)
axes[1].set_title("Phân phối lượng mưa, giới hạn tại phân vị 99%")
axes[1].set_xlabel("Lượng mưa (mm)")
axes[1].set_ylabel("Số quan sát")

plt.tight_layout()
plt.show()

In [ ]:
numeric_df = raw_df.select_dtypes(include=np.number)
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(12, 10))
image = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticklabels(corr.columns, fontsize=8)
ax.set_title("Ma trận tương quan giữa các biến số")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

display(
    corr[TARGET_COL]
    .drop(TARGET_COL)
    .sort_values(key=np.abs, ascending=False)
    .rename("correlation_with_rain")
    .to_frame()
    .head(15)
)

## 6. Kết luận khám phá dữ liệu

Sau khi chạy notebook, nhóm cần ghi nhận:

- Khoảng thời gian và số lượng quan sát.
- Tỷ lệ giá trị thiếu.
- Dữ liệu có được lấy mẫu đều 10 phút hay không.
- Tỷ lệ thời điểm có mưa.
- Các biến có tương quan đáng chú ý với lượng mưa.
- Các dấu hiệu ngoại lệ cần xử lý ở notebook tiếp theo.